# Get Akamai Control Center events

The Akamai Control Center (ACC), the central Akamai management platform, has an Event Viewer API endpoint that lets you look at activities (events) relating to ACC. Some examples of events include: configuration changes, login attempts, and log deliveries.

This endpoint can be used to retrieve these ACC events and store them in an object store or forward them to a logging system.

## Set Up EdgeGrid Session

Let's first set up an EdgeGrid session using information from the `~/.edgerc` file. The correct *section* can be set via `export AKAMAI_EDGEGRID_SECTION=section`; otherwise, `default` is going to be used.

Make sure your API user has the [Event-Viewer](https://techdocs.akamai.com/event-viewer/reference/api-get-started) permissions.

This notebook supports the `AccountSwitchKey` option, which can be set via `export AKAMAI_ACCOUNT_SWITCH_KEY="1-2345:A-BCDE"`

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests
from akamai.edgegrid import EdgeGridAuth, EdgeRc

edgerc_path = os.path.expanduser("~/.edgerc")

if not os.path.exists(edgerc_path):
    raise FileNotFoundError(f"Could not find .edgerc at {edgerc_path}")

section = os.getenv("AKAMAI_EDGEGRID_SECTION", "default")
account_switch_key = os.getenv("AKAMAI_ACCOUNT_SWITCH_KEY")

edgerc = EdgeRc(edgerc_path)
host = edgerc.get(section, "host")

session = requests.Session()
session.auth = EdgeGridAuth.from_edgerc(edgerc, section)
session.headers.update({"Accept": "application/json"})

base_url = f"https://{host}"
print(f"EdgeGrid session configured for host: {host}")

if account_switch_key:
    print(f"Using AccountSwitchKey: {account_switch_key}")

## Event Types

The [Event-Viewer API](https://techdocs.akamai.com/event-viewer/reference/get-event-types) supports different [event types](https://techdocs.akamai.com/event-viewer/docs/event-types-and-activities). Customers might only be interested in login events, for example, so we can filter on a specific event type.

Let's first get a list of all available event types, but only search for `All Logins`. You can add `Property Manager` to the list to also export those events.

In [ ]:
WANTED_EVENT_TYPES = ["All Logins"]

event_types_url = f"{base_url}/event-viewer-api/v1/event-types"
params = {"accountSwitchKey": account_switch_key} if account_switch_key else None

response = session.get(event_types_url, params=params, timeout=30)
response.raise_for_status()

event_types = [
    {
        "eventTypeId": event_type.get("eventTypeId"),
        "eventTypeName": event_type.get("eventTypeName"),
    }
    for event_type in response.json()
    if event_type.get("eventTypeName") in WANTED_EVENT_TYPES
]

print(json.dumps(event_types, indent=2))

## Get events

Now that we have the `eventTypeId` for `All Logins`, get the last **hour** of events for that type.
>When using `application/json` as content-type in the call, the default, there is a limit of 50 events per call.

*Only the first 2 events will be printed; there is no need to print them all.*

In [ ]:
events_url = f"{base_url}/event-viewer-api/v1/events"

start_time = datetime.now(timezone.utc).replace(microsecond=0) - timedelta(hours=1)

all_events = []
for event_type in event_types:
    params = {
        "eventTypeId": int(event_type["eventTypeId"]),
        "start": start_time.strftime("%Y-%m-%dT%H:%M:%S"),
        "limit": 50,
    }
    if account_switch_key:
        params["accountSwitchKey"] = account_switch_key

    page_events = []
    next_url = events_url
    next_params = params
    page_count = 0

    while next_url:
        response = session.get(next_url, params=next_params, timeout=30)
        if not response.ok:
            print(f"Request URL: {response.url}")
            print(f"Status: {response.status_code}")
            print("Error body:")
            print(response.text)
            response.raise_for_status()

        payload = response.json()
        events = payload.get("events", [])
        page_events.extend(events)
        page_count += 1

        next_link = next((link for link in payload.get("links", []) if link.get("rel") == "next"), None)
        if next_link and next_link.get("href"):
            next_url = f"{base_url}{next_link['href']}"
            next_params = None
            print(f"Found {len(events)} {event_type['eventTypeName']} events but more available, let's fetch them.")
            time.sleep(1)
        else:
            next_url = None

    print(
        f"{event_type['eventTypeName']} (id={event_type['eventTypeId']}): "
        f"{len(page_events)} events across {page_count} page(s)"
    )
    all_events.extend(page_events)

print(f"\nTotal events: {len(all_events)}")
print(f"Showing first {min(2, len(all_events))} events for debugging:")
print(json.dumps(all_events[:2], indent=2))

## Forward to external system

This `all_events` variable can now be forwarded to whatever logging tool you are using.